# Data Fusion — Data Science Salaries

Merges three CSV datasets into a single, normalised DataFrame:
- `salaries.csv` — 151 k rows, 2020–2025, coded categoricals, has `remote_ratio`
- `data_science_salaries.csv` — 6.6 k rows, 2020–2024, full-text categoricals, has `work_models`
- `DS.csv` — 3.8 k rows, 2020–2023, coded categoricals, no residence/remote columns

**Target schema (coded):**

| column | values |
|--------|--------|
| `work_year` | int |
| `experience_level` | EN / MI / SE / EX |
| `employment_type` | FT / PT / CT / FL |
| `job_title` | string |
| `salary` | int (original currency) |
| `salary_currency` | ISO-4217 |
| `salary_in_usd` | int |
| `employee_residence` | ISO-3166 alpha-2 (NaN where unavailable) |
| `remote_ratio` | 0 / 50 / 100 (NaN where unavailable) |
| `company_location` | ISO-3166 alpha-2 |
| `company_size` | S / M / L |
| `source` | origin file tag |

In [2]:
import pandas as pd
import numpy as np

DATA = 'data/'

## 1 · Load raw files

In [3]:
raw_salaries = pd.read_csv(DATA + 'salaries.csv')
raw_ds_sal   = pd.read_csv(DATA + 'data_science_salaries.csv')
raw_ds       = pd.read_csv(DATA + 'DS.csv')

print('salaries          :', raw_salaries.shape)
print('data_science_salaries:', raw_ds_sal.shape)
print('DS                :', raw_ds.shape)

salaries          : (151445, 11)
data_science_salaries: (6599, 11)
DS                : (3761, 9)


## 2 · Normalise `data_science_salaries.csv`

In [4]:
raw_salaries['remote_ratio'].unique()

array([  0, 100,  50])

In [5]:
# ── experience_level ──────────────────────────────────────────────────────────
exp_map = {
    'Entry-level':     'EN',
    'Mid-level':       'MI',
    'Senior-level':    'SE',
    'Executive-level': 'EX',
}

# ── employment_type ───────────────────────────────────────────────────────────
emp_map = {
    'Full-time':  'FT',
    'Part-time':  'PT',
    'Contract':   'CT',
    'Freelance':  'FL',
}

# ── company_size ──────────────────────────────────────────────────────────────
size_map = {
    'Small':  'S',
    'Medium': 'M',
    'Large':  'L',
}

# ── work_models → remote_ratio ────────────────────────────────────────────────
# Remote=100, On-site=0, Hybrid=50  (same semantics as the remote_ratio column)
work_model_map = {
    'Remote':  100,
    'On-site':   0,
    'Hybrid':   50,
}

# ── country name → ISO alpha-2 (covers values present in this dataset) ────────
country_map = {
    'United States': 'US', 'United Kingdom': 'GB', 'Canada': 'CA',
    'Germany': 'DE', 'France': 'FR', 'India': 'IN', 'Australia': 'AU',
    'Spain': 'ES', 'Brazil': 'BR', 'Netherlands': 'NL', 'Italy': 'IT',
    'Singapore': 'SG', 'Portugal': 'PT', 'Mexico': 'MX', 'Poland': 'PL',
    'Sweden': 'SE', 'Switzerland': 'CH', 'Denmark': 'DK', 'Norway': 'NO',
    'Belgium': 'BE', 'Austria': 'AT', 'Greece': 'GR', 'Romania': 'RO',
    'Czech Republic': 'CZ', 'Hungary': 'HU', 'Argentina': 'AR',
    'Colombia': 'CO', 'Chile': 'CL', 'Philippines': 'PH', 'Indonesia': 'ID',
    'Malaysia': 'MY', 'Thailand': 'TH', 'Vietnam': 'VN', 'Japan': 'JP',
    'South Korea': 'KR', 'China': 'CN', 'Israel': 'IL', 'Turkey': 'TR',
    'South Africa': 'ZA', 'Nigeria': 'NG', 'Kenya': 'KE', 'Egypt': 'EG',
    'United Arab Emirates': 'AE', 'Saudi Arabia': 'SA', 'Pakistan': 'PK',
    'Bangladesh': 'BD', 'Sri Lanka': 'LK', 'Ukraine': 'UA', 'Russia': 'RU',
    'New Zealand': 'NZ', 'Ireland': 'IE', 'Finland': 'FI',
}

ds_sal = raw_ds_sal.copy()
ds_sal['experience_level'] = ds_sal['experience_level'].map(exp_map)
ds_sal['employment_type']  = ds_sal['employment_type'].map(emp_map)
ds_sal['company_size']     = ds_sal['company_size'].map(size_map)
ds_sal['remote_ratio']     = ds_sal['work_models'].map(work_model_map)
ds_sal['employee_residence'] = ds_sal['employee_residence'].map(
    lambda x: country_map.get(x, x)  # keep value as-is if already a code
)
ds_sal['company_location'] = ds_sal['company_location'].map(
    lambda x: country_map.get(x, x)
)
ds_sal = ds_sal.drop(columns=['work_models'])

print('Unmapped experience_level:', ds_sal['experience_level'].isna().sum())
print('Unmapped employment_type :', ds_sal['employment_type'].isna().sum())
print('Unmapped company_size    :', ds_sal['company_size'].isna().sum())
print('Unmapped work_models     :', ds_sal['remote_ratio'].isna().sum())

Unmapped experience_level: 0
Unmapped employment_type : 0
Unmapped company_size    : 0
Unmapped work_models     : 0


## 3 · Align schemas and tag sources

In [6]:
TARGET_COLS = [
    'work_year', 'experience_level', 'employment_type', 'job_title',
    'salary', 'salary_currency', 'salary_in_usd',
    'employee_residence', 'remote_ratio',
    'company_location', 'company_size',
]

def align(df, source_tag):
    """Add missing columns (as NaN) and attach a source tag."""
    out = df.copy()
    for col in TARGET_COLS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[TARGET_COLS].copy()
    out['source'] = source_tag
    return out

df_salaries = align(raw_salaries, 'salaries')
df_ds_sal   = align(ds_sal,       'data_science_salaries')
df_ds       = align(raw_ds,       'DS')

print(df_salaries.shape, df_ds_sal.shape, df_ds.shape)

(151445, 12) (6599, 12) (3761, 12)


## 4 · Concatenate and de-duplicate

In [7]:
fused = pd.concat([df_salaries, df_ds_sal, df_ds], ignore_index=True)

print(f'Total rows before dedup : {len(fused):,}')

# Exact duplicates (ignoring source tag) are dropped; keep first occurrence
dedup_cols = TARGET_COLS  # all content columns
fused_dedup = fused.drop_duplicates(subset=dedup_cols, keep='first')

print(f'Total rows after  dedup : {len(fused_dedup):,}')
print(f'Duplicates removed      : {len(fused) - len(fused_dedup):,}')

Total rows before dedup : 161,805
Total rows after  dedup : 74,799
Duplicates removed      : 87,006


## 5 · Light cleaning

In [8]:
df = fused_dedup.copy()

# Enforce types
df['work_year']     = df['work_year'].astype(int)
df['salary']        = df['salary'].astype(int)
df['salary_in_usd'] = df['salary_in_usd'].astype(int)
df['remote_ratio']  = pd.to_numeric(df['remote_ratio'], errors='coerce')

# Normalise string columns
for col in ['experience_level', 'employment_type', 'company_size',
            'employee_residence', 'company_location', 'salary_currency']:
    df[col] = df[col].str.strip().str.upper()

df['job_title'] = df['job_title'].str.strip()

print(df.dtypes)
print()
print(df.isnull().sum())

work_year               int64
experience_level       object
employment_type        object
job_title              object
salary                  int64
salary_currency        object
salary_in_usd           int64
employee_residence     object
remote_ratio          float64
company_location       object
company_size           object
source                 object
dtype: object

work_year                0
experience_level         0
employment_type          0
job_title                0
salary                   0
salary_currency          0
salary_in_usd            0
employee_residence    2410
remote_ratio          2410
company_location         0
company_size             0
source                   0
dtype: int64


## 6 · Overview of the fused dataset

In [9]:
print('Shape :', df.shape)
print()
print('Year distribution:')
print(df['work_year'].value_counts().sort_index())
print()
print('Experience level:')
print(df['experience_level'].value_counts())
print()
print('Source breakdown:')
print(df['source'].value_counts())
print()
df.describe(include='all')

Shape : (74799, 12)

Year distribution:
work_year
2020      159
2021      481
2022     2226
2023     5918
2024    27733
2025    38282
Name: count, dtype: int64

Experience level:
experience_level
SE    39388
MI    24145
EN     8276
EX     2990
Name: count, dtype: int64

Source breakdown:
source
salaries                 71913
DS                        2410
data_science_salaries      476
Name: count, dtype: int64



,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,source
count,74799.000000,74799,74799,74799,7.479900e+04,74799,74799.000000,72389,72389.000000,74799,74799,74799
unique,NaN,4,4,437,NaN,26,NaN,141,NaN,125,3,3
top,NaN,SE,FT,Data Scientist,NaN,USD,NaN,US,NaN,US,M,salaries
freq,NaN,39388,73999,7665,NaN,68241,NaN,60421,NaN,62257,72347,71913
mean,2024.345366,NaN,NaN,NaN,1.639327e+05,NaN,150594.301675,NaN,24.671566,NaN,NaN,NaN
std,0.826180,NaN,NaN,NaN,3.240495e+05,NaN,77139.707318,NaN,42.965435,NaN,NaN,NaN
min,2020.000000,NaN,NaN,NaN,6.000000e+03,NaN,5132.000000,NaN,0.000000,NaN,NaN,NaN
25%,2024.000000,NaN,NaN,NaN,9.611650e+04,NaN,95734.500000,NaN,0.000000,NaN,NaN,NaN
50%,2025.000000,NaN,NaN,NaN,1.395000e+05,NaN,138343.000000,NaN,0.000000,NaN,NaN,NaN
75%,2025.000000,NaN,NaN,NaN,1.920000e+05,NaN,190000.000000,NaN,0.000000,NaN,NaN,NaN


## 7 · Save fused dataset

In [13]:
out_path = DATA + 'fused_salaries.csv'
df = df.drop('source', axis=1)
print(df.keys)
df.to_csv(out_path, index=False)
print(f'Saved {len(df):,} rows → {out_path}')

<bound method NDFrame.keys of         work_year experience_level employment_type                  job_title  \
0            2025               EX              FT               Head of Data   
1            2025               EX              FT               Head of Data   
2            2025               SE              FT             Data Scientist   
3            2025               SE              FT             Data Scientist   
4            2025               MI              FT                   Engineer   
...           ...              ...             ...                        ...   
161800       2020               SE              FT   Principal Data Scientist   
161801       2020               SE              FT             Data Scientist   
161802       2020               SE              FT       Data Science Manager   
161803       2020               SE              FT  Machine Learning Engineer   
161804       2020               SE              FT             Data Scientist  

In [3]:
print(len("O artigo [1] traz insights através dos dados referentes aos salários em data science mundialmente. Muito interessante ler esse artigo, pois traz 13 perguntas de pesquisa que são respondidas através de visualização de dados, inspirando possíveis RQs e maneiras de respondê-las através da visualização dos dados. Problemas identificados no artigo: título refere 2021-2023 e a RQ1 responde uma pergunta de 2020-2023; não explicam como foi feito o tratamento de dados; não explicam o dataset de forma adequada e usam somente variantes de gráficos de barra e de linha. O artigo [2] é útil como base contextual para o estudo de salários em data science, mesmo não sendo centrado em visualização de dados. Seu principal mérito está em mostrar que o trabalho remoto, isoladamente, explica pouco da variação salarial, mas ganha relevância quando analisado junto de variáveis de controle como nível de experiência e tamanho da empresa. Isso é valioso porque desloca o foco de leituras puramente temporais e abre espaço para explorar relações multivariadas e padrões estruturais menos intuitivos. Assim, o artigo é mais forte como ponto de partida analítico do que como modelo definitivo. Para o nosso trabalho, ele é relevante justamente por ampliar o repertório de perguntas que as visualizações podem responder. O artigo [3] apresenta um sistema de visualização interativa que explora anúncios de emprego com foco nas habilidades exigidas. Ele é útil para o trabalho porque mostra como relacionar variáveis diversas em diferentes visualizações. O estudo usa cerca de 2,43 milhões de anúncios, focados em vagas de computação. Seu principal mérito é propor uma abordagem skill-driven, permitindo filtrar vagas por similaridade de habilidades, analisar clusters e comparar anúncios específicos. Como limitações, o artigo é restrito ao mercado chinês. Para o nosso trabalho, serve como inspiração para criar RQs e visualizações que relacionem salários a variáveis estruturais do mercado de data science."))

1991
